# 07 - Owner Dashboard

**Inputs:** `salon_clean.csv`, `tiers.json`, `service_clusters.csv`, `cluster_recommendations.csv`,
`scenario_summary.csv`, `scenarios.py`, `pricing.py`

**Output:** `outputs/owner_dashboard.html` (opens offline in any browser)

Shows revenue index by hour for an average week, flat vs tiered pricing,
for each policy and service cluster under all three scenarios. Uses the same pricing
model as notebook 06/ pricing.py.

In [6]:
import sys, json, os; sys.path.append("..")
import numpy as np, pandas as pd
from scenarios import SCENARIOS
from pricing import apply_pricing, revenue

s = pd.read_csv("../data/processed/salon_clean.csv")
TIER = {int(k): v for k, v in json.load(open("../outputs/tables/tiers.json")).items()}
clus = pd.read_csv("../outputs/tables/service_clusters.csv", index_col=0)
CLUSTER_OF = clus["cluster"].to_dict()

b = s[s["hour_valid"]].copy()
b["appt_hour"] = b["appt_hour"].astype(int)
b["tier"] = b["appt_hour"].map(TIER)
b["cluster"] = b["service_group"].map(CLUSTER_OF).astype(int)

n_weeks = int(s["week_index"].max()) + 1
HOURS = list(range(9, 18))

# Avg bookings per week
hourly = (b.groupby(["cluster", "appt_hour"]).size().unstack(fill_value=0)
            .reindex(columns=HOURS, fill_value=0) / n_weeks)

# Median days booked ahead by demand tier
lead_by_tier = b.groupby("tier")["lead_days"].median()

print("Average bookings per week by cluster:")
print(hourly.sum(axis=1).round(2))
print("\nMedian lead days by tier:")
print(lead_by_tier)

Average bookings per week by cluster:
cluster
-1    0.11
 0    1.25
 1    0.48
 2    1.48
 3    0.89
dtype: float64

Median lead days by tier:
tier
mid     16.0
off      8.0
peak    20.5
Name: lead_days, dtype: float64


In [7]:
# Identifable names and wording for relevant cluster
members = {c: list(clus.index[clus["cluster"] == c]) for c in range(4)}
# stops if cluster numbers ever change, so text does not lands in wrong group
assert "Women's Haircut" in members[0] and "Mini Custom Highlight" in members[1]
assert "Gloss with Haircut" in members[2] and "All Over Color" in members[3]

GROUP_TEXT = {
    3: {"name": "Color services",
        "services": "All over color, full balayage, root touch-up with cut",
        "finding": "Six in ten of these bookings land in your busiest hours, and clients rarely cancel.",
        "action": "a 10% busy-hour increase. This is where it makes the biggest difference."},
    2: {"name": "Planned-ahead services",
        "services": "Partial highlights, gloss with haircut, partial balayage, and other services",
        "finding": "Clients book about 4 weeks ahead, so most can pick another time if busy hours cost more.",
        "action": "a modest busy-hour increase. Expect clients to change times rather than leave."},
    1: {"name": "Highlights",
        "services": "Full and mini custom highlights",
        "finding": "Most of these are already booked at quieter times.",
        "action": "featuring these in quiet-time offers to fill empty afternoon slots."},
    0: {"name": "Haircuts and blonding",
        "services": "Women's haircut, signature blonde session",
        "finding": "Your most booked services, but about 1 in 4 cancel or no-show.",
        "action": "a deposit or reminder text first. A busy-hour increase adds only a little here."},
}
# Nums behind the wording
print(clus[clus["cluster"] >= 0].groupby("cluster")[["peak_share", "median_lead", "cancel_rate"]].mean().round(2))

         peak_share  median_lead  cancel_rate
cluster                                      
0              0.47         9.25         0.23
1              0.22         7.75         0.00
2              0.39        28.62         0.12
3              0.62         8.00         0.03


In [8]:
# Page template: HTML page with Plotly chart
PAGE_TEMPLATE = r"""<!doctype html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Pricing Your Busy Hours</title>
<style>
  :root { --ink:#1f2328; --muted:#5b6470; --bg:#faf8f5; --card:#ffffff; --line:#e6e1da;
          --now:#b8b2aa; --new:#2f6f8f; --busy:#c0592b; --quiet:#2f6f8f; --steady:#8a8178; --good:#2e7d4f; --bad:#b03a2e; }
  * { box-sizing:border-box; }
  body { margin:0; background:var(--bg); color:var(--ink); font:16px/1.55 -apple-system, "Segoe UI", Roboto, Arial, sans-serif; }
  main { max-width:960px; margin:0 auto; padding:32px 16px 48px; }
  h1 { font-size:30px; line-height:1.2; margin:0 0 8px; }
  h2 { font-size:22px; margin:40px 0 12px; }
  h3 { font-size:17px; margin:0 0 4px; }
  p { margin:0 0 10px; }
  .lead { font-size:18px; color:var(--muted); max-width:680px; }
  .grid3 { display:grid; grid-template-columns:repeat(auto-fit,minmax(220px,1fr)); gap:12px; margin-top:20px; }
  .card, .svc, .tool { background:var(--card); border:1px solid var(--line); border-radius:12px; padding:16px 18px; }
  .card .k { font-size:13px; text-transform:uppercase; letter-spacing:.04em; color:var(--muted); }
  .card .v { font-size:20px; font-weight:600; margin-top:4px; }
  .busy { color:var(--busy); } .quiet { color:var(--quiet); }
  .tool { display:grid; grid-template-columns:280px 1fr; gap:24px; padding:20px; }
  @media (max-width:760px) { .tool { grid-template-columns:1fr; } }
  label { display:block; font-weight:600; font-size:14px; margin:14px 0 6px; }
  label:first-child { margin-top:0; }
  select, input[type=number] { width:100%; font:inherit; padding:8px 10px; border:1px solid var(--line); border-radius:8px; background:#fff; }
  input[type=range] { width:100%; accent-color:var(--new); }
  .val { float:right; font-weight:600; color:var(--new); }
  .hint { font-size:13px; color:var(--muted); margin-top:4px; }
  .result .big { font-size:34px; font-weight:700; line-height:1.1; }
  .result .sub { color:var(--muted); margin-top:6px; }
  .moves { display:flex; flex-wrap:wrap; gap:8px 20px; margin:14px 0 4px; font-size:15px; }
  .tip { background:#f3efe9; border-radius:8px; padding:10px 12px; font-size:14px; margin-top:12px; display:none; }
  ol.steps { padding-left:20px; margin:0; } ol.steps li { margin-bottom:10px; }
  .svcgrid { display:grid; grid-template-columns:repeat(2,1fr); gap:12px; }
  @media (max-width:640px) { .svcgrid { grid-template-columns:1fr; } }
  .svc .members { font-size:13px; color:var(--muted); }
  .svc .todo { margin:0; }
  details { margin-top:40px; background:var(--card); border:1px solid var(--line); border-radius:12px; padding:14px 18px; }
  summary { cursor:pointer; font-weight:600; }
  details ul { color:var(--muted); font-size:14px; padding-left:20px; }
</style>
<script>__PLOTLYJS__</script>
</head>
<body><main>

<h1>Your busiest hours could earn you more</h1>
<p class="lead">This page uses almost a year of real appointments from one stylist as an example. Some hours filled up
again and again while others sat half-empty. Charging a little more when you are in demand can raise your income
without losing many clients. Your own busy and quiet times may be different, so use this as a guide.</p>

<div class="grid3">
  <div class="card"><div class="k">Busiest times in this business</div>div class="v busy">__BUSY__</div></div>
  <div class="card"><div class="k">Quietest times in this business</div><div class="v quiet">__QUIET__</div></div>
  <div class="card"><div class="k">How far ahead clients booked with this salon owners business</div>
    <div class="v">Busy hours: __LEAD_BUSY__<br>Quiet hours: __LEAD_QUIET__</div></div>
</div>

<h2>Try it yourself</h2>
<div class="tool">
  <div>
    <label for="grp">Which services?</label>
    <select id="grp"></select>
    <label for="raise">Raise prices at busy times <span class="val" id="raiseV"></span></label>
    <input type="range" id="raise" min="0" max="20" step="5" value="10">
    <label for="disc">Lower prices at quiet times <span class="val" id="discV"></span></label>
    <input type="range" id="disc" min="0" max="20" step="5" value="0">
    <label for="price">Your average price for these services</label>
    <input type="number" id="price" min="1" step="5" value="100">
    <div class="hint">Use your own number. It only changes the dollar amounts.</div>
    <label for="scn">How cautious should the estimate be?</label>
    <select id="scn">
      <option value="conservative">Cautious (clients react strongly)</option>
      <option value="base" selected>Most likely</option>
      <option value="optimistic">Hopeful (clients react mildly)</option>
    </select>
  </div>
  <div class="result" aria-live="polite">
    <div class="big" id="big"></div>
    <div class="sub" id="range"></div>
    <div class="moves" id="moves"></div>
    <div class="hint" style="margin-top:12px">Bookings per month at each hour</div>
    <div id="chart" style="width:100%;height:280px" role="img" aria-label="Bookings per month by hour, today and with new prices"></div>
    <div class="tip" id="tip"></div>
  </div>
</div>

<h2>What to do next</h2>
<div class="card"><ol class="steps">
  <li><b>Start small.</b> Add about 10% to color services at __BUSY__ for 6 to 8 weeks.
      Keep a few busy slots at your usual price so regulars still have a choice.</li>
  <li><b>Fill quiet times with short offers, not permanent discounts.</b> A limited-time deal for
      __QUIET__ (for example, "book by Friday") moves clients without lowering what everyone else pays.</li>
  <li><b>Cut no-shows on haircuts.</b> About 1 in 4 haircut and blonding bookings cancel or no-show.
      A small deposit or a reminder text will likely protect more income than a price change.</li>
</ol></div>

<h2>By service</h2>
<div class="svcgrid">__CARDS__</div>

<details><summary>How these estimates work</summary>
<ul>
  <li>Busy times, quiet times, and booking lead times come from this one salon's history. To find your own, count
      your bookings by hour over the past few months: the hours that fill most often are your busy times.</li>
  <li>Based on 186 appointments from one stylist's booking log over about 44 weeks (White, 2025).</li>
  <li>How clients react to price changes comes from a published study of off-peak pricing
      (Thommen &amp; Hintermann, 2023), not from this salon. The range shows less and more flexible clients.</li>
  <li>When busy-hour prices go up, a few clients book elsewhere and about half of them move to a quieter time instead.</li>
  <li>Tested on the last 13 weeks of the log: a 20% busy-hour increase raised income by about 3% to 5%,
      and every version beat flat pricing.</li>
  <li>Monthly numbers are weekly averages times 4.3. Small salons vary a lot week to week, so treat these as a guide
      and confirm with a short trial.</li>
</ul></details>

</main>
<script>
const D = __DATA__;
const $ = id => document.getElementById(id);
const MONTH = 52 / 12;
const TIER_WORD = {peak: "busy", mid: "steady", off: "quiet"};

D.groups.forEach(g => { const o = document.createElement("option"); o.value = g.key; o.textContent = g.name; $("grp").appendChild(o); });

function calc(clusters, p, d, sc) {
  const n = D.hours.length, now = Array(n).fill(0), next = Array(n).fill(0);
  let revNow = 0, revNew = 0, busyNow = 0, busyNew = 0, quietNow = 0, quietNew = 0;
  const nOff = D.hours.filter(h => h.tier === "off").length;
  clusters.forEach(c => {
    const a = D.perWeek[c];
    let pk = 0, md = 0, of = 0;
    a.forEach((v, i) => { const t = D.hours[i].tier; if (t === "peak") pk += v; else if (t === "off") of += v; else md += v; });
    const lost = pk * sc.ep * p, shifted = lost * sc.sh, gained = of * sc.ed * d;
    const npk = pk - lost, nof = of + gained + shifted;
    a.forEach((v, i) => {
      const t = D.hours[i].tier;
      now[i] += v;
      next[i] += t === "peak" ? (pk ? v * npk / pk : 0) : t === "off" ? (of ? v * nof / of : nof / nOff) : v;
    });
    revNow += pk + md + of;
    revNew += npk * (1 + p) + md + nof * (1 - d);
    busyNow += pk; busyNew += npk; quietNow += of; quietNew += nof;
  });
  return {now, next, revNow, revNew, busyNow, busyNew, quietNow, quietNew};
}

const money = x => (x < 0 ? "\u2212$" : "+$") + Math.abs(Math.round(x)).toLocaleString();
const one = x => (Math.round(x * 10) / 10).toFixed(1);

const HEX = {now: "#b8b2aa", new: "#2f6f8f", peak: "#c0592b", mid: "#8a8178", off: "#2f6f8f"};
function draw(now, next, scnLabel) {
  const x = D.hours.map(h => h.label);
  const ticks = D.hours.map(h => `${h.label}<br><span style="color:${HEX[h.tier]}">${TIER_WORD[h.tier]}</span>`);
  const m = v => v.map(z => Math.round(z * MONTH * 10) / 10);
  Plotly.react("chart", [
    {type: "bar", name: "Today", x, y: m(now), marker: {color: HEX.now},
     hovertemplate: "%{x}<br>Today: %{y:.1f} bookings a month<extra></extra>"},
    {type: "bar", name: "With new prices", x, y: m(next), marker: {color: HEX.new},
     hovertemplate: "%{x}<br>With new prices: %{y:.1f} bookings a month<extra>" + scnLabel + "</extra>"},
  ], {
    barmode: "group", bargap: 0.25, margin: {l: 40, r: 10, t: 10, b: 60},
    paper_bgcolor: "#ffffff", plot_bgcolor: "#ffffff",
    font: {family: "-apple-system, Segoe UI, Roboto, Arial, sans-serif", size: 12, color: "#1f2328"},
    xaxis: {tickvals: x, ticktext: ticks, fixedrange: true},
    yaxis: {gridcolor: "#ece7e0", zeroline: false, fixedrange: true, rangemode: "tozero"},
    legend: {orientation: "h", x: 0, y: -0.28, font: {size: 13, color: "#5b6470"}},
  }, {displayModeBar: false, responsive: true});
}

function update() {
  const g = D.groups.find(x => x.key === $("grp").value);
  const p = +$("raise").value / 100, d = +$("disc").value / 100;
  const price = Math.max(0, +$("price").value || 0);
  $("raiseV").textContent = p ? `+${p * 100}%` : "no change";
  $("discV").textContent = d ? `\u2212${d * 100}%` : "no change";

  const res = {};
  for (const k in D.scenarios) res[k] = calc(g.clusters, p, d, D.scenarios[k]);
  const sel = $("scn").value, base = res[sel];
  const scnLabel = $("scn").selectedOptions[0].textContent;
  const gain = k => (res[k].revNew - res[k].revNow) * price * MONTH;
  const pct = base.revNow ? (base.revNew - base.revNow) / base.revNow * 100 : 0;

  if (!p && !d) {
    $("big").textContent = "Same as today";
    $("range").textContent = "Move a slider to see what changes.";
  } else {
    const gs = Object.keys(res).map(gain), lo = Math.min(...gs), hi = Math.max(...gs);
    $("big").innerHTML = `${money(gain(sel))} a month <span style="font-size:18px;color:${pct >= 0 ? "var(--good)" : "var(--bad)"}">(${pct >= 0 ? "+" : "\u2212"}${Math.abs(pct).toFixed(1)}%)</span>`;
    $("range").textContent = `Likely between ${money(lo)} and ${money(hi)} a month, depending on how your clients react.`;
  }
  $("moves").innerHTML =
    `<span><b class="busy">Busy-hour</b> bookings: ${one(base.busyNow * MONTH)} \u2192 ${one(base.busyNew * MONTH)} a month</span>` +
    `<span><b class="quiet">Quiet-hour</b> bookings: ${one(base.quietNow * MONTH)} \u2192 ${one(base.quietNew * MONTH)} a month</span>`;
  draw(base.now, base.next, scnLabel);

  const tips = [];
  if (d > 0) tips.push("Discounts also lower the price for clients who would have booked anyway, so they usually cost more than they bring in. Use them for specific slots you need to fill.");
  if (p > 0.10) tips.push("Bigger increases look better on paper, but regulars are more likely to notice. Hotels change weekday rates by only about 6 to 7%.");
  $("tip").style.display = tips.length ? "block" : "none";
  $("tip").textContent = tips.join(" ");
}
["grp", "raise", "disc", "price", "scn"].forEach(id => $(id).addEventListener("input", update));
update();
</script>
</body></html>"""
print("Template ready:", len(PAGE_TEMPLATE), "characters")

Template ready: 11779 characters


In [9]:
# Owner dashboard: data + explaination text all in one HTML file
from plotly.offline import get_plotlyjs
HOUR_LABEL = {9: "9 am", 10: "10 am", 11: "11 am", 12: "12 pm", 13: "1 pm",
              14: "2 pm", 15: "3 pm", 16: "4 pm", 17: "5 pm"}

busy_h = [HOUR_LABEL[h] for h in HOURS if TIER[h] == "peak"]
quiet_h = [HOUR_LABEL[h] for h in HOURS if TIER[h] == "off"]
weeks = lambda d: f"about {max(1, round(d / 7))} week" + ("" if round(d / 7) <= 1 else "s")

page_data = {
    "hours": [{"label": HOUR_LABEL[h], "tier": TIER[h]} for h in HOURS],
    "groups": [{"key": "all", "name": "All services",
                "clusters": [str(c) for c in GROUP_TEXT]}]
              + [{"key": str(c), "name": t["name"], "clusters": [str(c)]} for c, t in GROUP_TEXT.items()],
    "perWeek": {str(c): [round(float(hourly.loc[c, h]), 4) for h in HOURS] for c in GROUP_TEXT},
    "scenarios": {k: {"ep": abs(v["e_prem"]), "ed": abs(v["e_disc"]), "sh": v["shift_share"]}
                  for k, v in SCENARIOS.items()},
}

cards_html = "".join(
    f'<div class="svc"><h3>{t["name"]}</h3><p class="members">{t["services"]}</p>'
    f'<p>{t["finding"]}</p><p class="todo"><b>Try:</b> {t["action"]}</p></div>'
    for t in GROUP_TEXT.values())

page = PAGE_TEMPLATE
for key, val in {
    "__DATA__": json.dumps(page_data),
    "__BUSY__": ", ".join(busy_h),
    "__QUIET__": ", ".join(quiet_h),
    "__LEAD_BUSY__": weeks(lead_by_tier["peak"]),
    "__LEAD_QUIET__": weeks(lead_by_tier["off"]),
    "__CARDS__": cards_html,
    "__PLOTLYJS__": get_plotlyjs(),          # embeds Plotly so the page works offline
}.items():
    page = page.replace(key, val)

os.makedirs("../outputs", exist_ok=True)
path = "../outputs/owner_dashboard.html"
with open(path, "w", encoding="utf-8") as f:
    f.write(page)
print(f"Saved {path} ({os.path.getsize(path) / 1e6:.1f} MB)")

Saved ../outputs/owner_dashboard.html (4.8 MB)


In [10]:
# Check: page default result computed in Python
p, d, price, MONTH = 0.10, 0.0, 100, 52 / 12
for name, sc in SCENARIOS.items():
    now = new = 0.0
    for c in GROUP_TEXT:
        row = hourly.loc[c]
        pk, md, of = (row[[h for h in HOURS if TIER[h] == t]].sum() for t in ["peak", "mid", "off"])
        now += revenue(pk, md, of)
        new += revenue(*apply_pricing(pk, md, of, p, d, sc["e_prem"], sc["e_disc"], sc["shift_share"]), p, d)
    print(f"{name:12s}: {(new - now) * price * MONTH:+.0f} dollars a month ({(new - now) / now:+.1%})")
print("\nOpen the page: each estimate option should match the line above.")
print("(Cautious = conservative, Most likely = base, Hopeful = optimistic)")

# B6: every service group x scenario returns bookings for page
for g in [list(GROUP_TEXT)] + [[c] for c in GROUP_TEXT]:
    for name in SCENARIOS:
        assert sum(hourly.loc[c].sum() for c in g) > 0
print(f"All {1 + len(GROUP_TEXT)} service options x {len(SCENARIOS)} estimate options return results")

conservative: +33 dollars a month (+1.9%)
base        : +45 dollars a month (+2.5%)
optimistic  : +51 dollars a month (+2.9%)

Open the page: each estimate option should match the line above.
(Cautious = conservative, Most likely = base, Hopeful = optimistic)
All 5 service options x 3 estimate options return results


## Insight
- Dashboard saved as `outputs/owner_dashboard.html`: easy to understand page for potential appointment-based business owners

- Interactive tool for service group, busy-hour increase (0-20%), quiet-hour discount (0-20%), owner's own average price

- Shows estimated monthly income change (base) with a range across conservative/optimistic scenarios,
  bookings moved between busy and quiet hours, and an hour-by-hour chart

- Page math verified against `pricing.py` (python check to ensure matches the page default result)

- Every service group option loads with data (B6)